In [1]:
# Code for 401(k) example in Lecture notes 1 illustrating tree-based methods

######################### Libraries
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
import seaborn as sns
from sklearn.metrics import r2_score
from sklearn.inspection import PartialDependenceDisplay
import os

######################### Options
dpl = False     # switch to save plots. Set to False to just run code
#plotdir = os.path.abspath("../Slides/figures")

# Read data
#######################################################
data_path = os.path.join("..", "Data", "restatw.dat")
data401k = pd.read_csv(data_path, sep=r"\s+")

# Variables we care about are
# net total financial assets (net_tfa)
# 401(k) participation (p401)
# 401(k) eligibility (e401)
# income (inc)
# age (age)
# family size (fsize)
# years schooling (educ)
# marital status (marr)
# gender (male)
# two-earner household (twoearn)

# Just keep the variables we will actually use in this example
data401k = data401k[['net_tfa', 'e401', 'age', 'inc', 'educ', 'fsize', 'marr', 'twoearn', 'db', 'pira', 'hown']]
print(data401k.head())

#########################################
# Set up training and testing sample

#########################################
np.random.seed(8261977)
ntrain = 8000    # number of training observations
tr = np.random.choice(len(data401k), ntrain, replace=False)  # draw ntrain observations from original data
train = data401k.iloc[tr]   # Training sample
test = data401k.iloc[~data401k.index.isin(tr)]   # Testing sample
nTr = len(train)
nTe = len(test)

##########################################
# Illustrate regression tree methods on 401(k) data

##########################################
# Just fit a simple tree for illustration.
# No cross-validation or tuning
X_train = train.drop('net_tfa', axis=1)
y_train = train['net_tfa']
X_test = test.drop('net_tfa', axis=1)
y_test = test['net_tfa']

tree3 = DecisionTreeRegressor(max_depth=3, min_impurity_decrease=0.0001, random_state=42)
tree3.fit(X_train, y_train)


if dpl:
    filenm = os.path.join(plotdir, 'Tree1.png')
    plt.figure(figsize=(10, 8))
    plot_tree(tree3, feature_names=X_train.columns, filled=True, rounded=True)
    plt.title("Decision Tree (max_depth=3)")
    plt.savefig(filenm, dpi=100, bbox_inches='tight')
    plt.close()



if dpl:
    filenm = os.path.join(plotdir, 'TreeFit.png')
    fig, ax = plt.subplots(figsize=(12, 6))
    PartialDependenceDisplay.from_estimator(tree3, X_train, features=X_train.columns, ax=ax)
    plt.suptitle("Partial Dependence Plots (Tree Fit)", fontsize=16)
    plt.savefig(filenm, dpi=100, bbox_inches='tight')
    plt.close()

if dpl:
    filenm = os.path.join(plotdir, 'TreeBins.png')

    # Get top 2 important features for 2D visualization
    importance_df = pd.DataFrame({
        'feature': X_train.columns,
        'importance': tree3.feature_importances_
    }).sort_values(by='importance', ascending=False)
    f1, f2 = importance_df['feature'].iloc[0], importance_df['feature'].iloc[1]

    # Create a meshgrid covering the range of the top 2 features
    x_min, x_max = X_train[f1].min(), X_train[f1].max()
    y_min, y_max = X_train[f2].min(), X_train[f2].max()
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    
    # Create a grid DataFrame where other features are set to their mean
    grid_df = pd.DataFrame({f1: xx.ravel(), f2: yy.ravel()})
    for col in X_train.columns:
        if col not in grid_df.columns:
            grid_df[col] = X_train[col].mean()  # Hold other variables constant

    # Predict net_tfa on the grid
    preds = tree3.predict(grid_df).reshape(xx.shape)

    # Plot heatmap using contourf:
    # This is used to replicate plotmo(type2 = "image") in R
    # It visualizes how the decision tree partitions the 2D space
    # Color intensity shows predicted values in each region ("bin")
    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, preds, cmap='YlOrRd', levels=100)

    # Overlay training data to give visual reference
    plt.scatter(X_train[f1], X_train[f2], c=y_train, cmap='viridis', edgecolor='k', s=15)
    plt.xlabel(f1)
    plt.ylabel(f2)
    plt.title('Tree Partition Visualization (like plotmo type2=image)')
    plt.colorbar(label='Predicted net_tfa')
    plt.savefig(filenm, dpi=100, bbox_inches='tight')
    plt.close()




# Big Tree
tree4 = DecisionTreeRegressor(max_depth=15, min_impurity_decrease=0.00001, random_state=42)
tree4.fit(X_train, y_train)

if dpl:
    filenm = os.path.join(plotdir, 'Tree4.pdf')
    plt.figure(figsize=(20, 20))
    plot_tree(tree4, feature_names=X_train.columns, filled=True, rounded=True)
    plt.savefig(filenm, bbox_inches='tight')
    plt.close()

# Two more trees for comparison
tree1 = DecisionTreeRegressor(max_depth=1, min_impurity_decrease=0.0001, random_state=42)
tree1.fit(X_train, y_train)

tree2 = DecisionTreeRegressor(max_depth=2, min_impurity_decrease=0.0001, random_state=42)
tree2.fit(X_train, y_train)

# In-sample and out-of-sample fit
def calculate_r2_manual(y_true, y_pred, y_train_mean):
    """Calculate R² manually using the formula from R code"""
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_train_mean) ** 2)
    return 1 - (ss_res / ss_tot)

y_train_mean = np.mean(y_train)

# In-sample R²
r2 = [
    calculate_r2_manual(y_train, tree1.predict(X_train), y_train_mean),
    calculate_r2_manual(y_train, tree2.predict(X_train), y_train_mean),
    calculate_r2_manual(y_train, tree3.predict(X_train), y_train_mean),
    calculate_r2_manual(y_train, tree4.predict(X_train), y_train_mean)
]

# Out-of-sample R² (Validation)
r2V = [
    calculate_r2_manual(y_test, tree1.predict(X_test), y_train_mean),
    calculate_r2_manual(y_test, tree2.predict(X_test), y_train_mean),
    calculate_r2_manual(y_test, tree3.predict(X_test), y_train_mean),
    calculate_r2_manual(y_test, tree4.predict(X_test), y_train_mean)
]

# Create results table
r2_table = np.zeros((4, 2))
r2_table[:, 0] = r2
r2_table[:, 1] = r2V

r2_df = pd.DataFrame(r2_table, 
                     index=["Depth 1", "Depth 2", "Depth 3", "Big"],
                     columns=["in-sample", "Validation"])

print(r2_df.round(3))

########################## Simple tree (with all variables)
## Use CV built into rpart

# Big Tree
from sklearn.model_selection import cross_val_score, GridSearchCV
import warnings
warnings.filterwarnings('ignore')

tree_cv = DecisionTreeRegressor(max_depth=15, min_impurity_decrease=0.00001, random_state=42)
# Use GridSearchCV to mimic rpart's built-in CV with different cp values
cp_values = np.logspace(-6, -1, 20)  # Range of complexity parameters
param_grid = {'ccp_alpha': cp_values}
grid_search = GridSearchCV(tree_cv, param_grid, cv=5, scoring='neg_mean_squared_error', return_train_score=True)
grid_search.fit(X_train, y_train)

# Extract CV results
fit_tree_cv = grid_search.cv_results_
cv_scores = -fit_tree_cv['mean_test_score']
cv_std = fit_tree_cv['std_test_score']
train_scores = -fit_tree_cv['mean_train_score']

# Convert to R² equivalent (1 - normalized MSE)
y_train_var = np.var(y_train)
cv_r2_scores = 1 - (cv_scores / y_train_var)
train_r2_scores = 1 - (train_scores / y_train_var)

# plot cv function
if dpl:
    filenm = os.path.join(plotdir, 'CVTree.png')
    plt.figure(figsize=(10, 6))
    x_axis = range(1, len(cv_r2_scores) + 1)
    plt.plot(x_axis, 1 - cv_r2_scores, 'black', linestyle='-', label='CV Error')
    plt.plot(x_axis, 1 - cv_r2_scores - cv_std/y_train_var, 'red', linestyle='-')
    plt.plot(x_axis, 1 - cv_r2_scores + cv_std/y_train_var, 'red', linestyle='-')
    plt.axhline(y=min(1 - cv_r2_scores), color='black', linestyle='--')
    plt.plot(x_axis, 1 - train_r2_scores, 'blue', linestyle='-', label='Training Error')
    plt.ylim(min(1 - cv_r2_scores) - 0.01, 1.01)
    plt.xlabel('Complexity Parameter')
    plt.ylabel('1 - R²')
    plt.xticks(x_axis, [f'{cp:.5f}' for cp in cp_values])
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(filenm, dpi=100, bbox_inches='tight')
    plt.close()

# Get CV min tree
bestcp = grid_search.best_params_['ccp_alpha']
pruned_tree = DecisionTreeRegressor(max_depth=15, ccp_alpha=bestcp, random_state=42)
pruned_tree.fit(X_train, y_train)

# R² in validation sample
r2V_pruned = calculate_r2_manual(y_test, pruned_tree.predict(X_test), y_train_mean)
print(f"Validation R² for pruned tree: {r2V_pruned}")

# Plot CV min tree
if dpl:
    filenm = os.path.join(plotdir, 'CVminTree.pdf')
    plt.figure(figsize=(15, 10))
    plot_tree(pruned_tree, feature_names=X_train.columns, filled=True, rounded=True)
    plt.savefig(filenm, bbox_inches='tight')
    plt.close()

##############################################################
# Random forest

##############################################################
# Random forest with default settings and OOB enabled
rForest_default = RandomForestRegressor(random_state=42, oob_score=True, bootstrap=True)
rForest_default.fit(X_train, y_train)

if dpl:
    filenm = os.path.join(plotdir, 'RFdefault.png')
    plt.figure(figsize=(10, 6))
    # Plot OOB error evolution (approximation since sklearn doesn't store all iterations)
    plt.plot(range(1, rForest_default.n_estimators + 1), 
             [rForest_default.oob_score_] * rForest_default.n_estimators)
    plt.title("Asset Random Forest - Default Settings")
    plt.xlabel("Number of Trees")
    plt.ylabel("OOB Score")
    plt.savefig(filenm, dpi=100, bbox_inches='tight')
    plt.close()

r2V_rForest_default = calculate_r2_manual(y_test, rForest_default.predict(X_test), y_train_mean)
print(f"Validation R² for random forest with default settings: {r2V_rForest_default}")

# Let's play with some tuning choices
# Don't randomize over variables
rForest_allx = RandomForestRegressor(max_features=X_train.shape[1], random_state=42, oob_score=True, bootstrap=True)
rForest_allx.fit(X_train, y_train)
r2V_rForest_allx = calculate_r2_manual(y_test, rForest_allx.predict(X_test), y_train_mean)
print(f"Validation R² for random forest with no x randomization: {r2V_rForest_allx}")

# Restrict minimum node size to 30
rForest_30 = RandomForestRegressor(min_samples_leaf=30, random_state=42, oob_score=True, bootstrap=True)
rForest_30.fit(X_train, y_train)
r2V_rForest_30 = calculate_r2_manual(y_test, rForest_30.predict(X_test), y_train_mean)
print(f"Validation R² for random forest with min node 30: {r2V_rForest_30}")

# Restrict minimum node size to 60
rForest_60 = RandomForestRegressor(min_samples_leaf=60, random_state=42, oob_score=True, bootstrap=True)
rForest_60.fit(X_train, y_train)
r2V_rForest_60 = calculate_r2_manual(y_test, rForest_60.predict(X_test), y_train_mean)
print(f"Validation R² for random forest with min node 60: {r2V_rForest_60}")

# Restrict minimum node size to 120
rForest_120 = RandomForestRegressor(min_samples_leaf=120, random_state=42, oob_score=True, bootstrap=True)
rForest_120.fit(X_train, y_train)
r2V_rForest_120 = calculate_r2_manual(y_test, rForest_120.predict(X_test), y_train_mean)
print(f"Validation R² for random forest with min node 120: {r2V_rForest_120}")

# Restrict minimum node size to 240
rForest_240 = RandomForestRegressor(min_samples_leaf=240, random_state=42, oob_score=True, bootstrap=True)
rForest_240.fit(X_train, y_train)
r2V_rForest_240 = calculate_r2_manual(y_test, rForest_240.predict(X_test), y_train_mean)
print(f"Validation R² for random forest with min node 240: {r2V_rForest_240}")

# Table of results
rf_table = np.zeros((6, 2))
rf_table[0, :] = [rForest_default.oob_score_, r2V_rForest_default]
rf_table[1, :] = [rForest_allx.oob_score_, r2V_rForest_allx]
rf_table[2, :] = [rForest_30.oob_score_, r2V_rForest_30]
rf_table[3, :] = [rForest_60.oob_score_, r2V_rForest_60]
rf_table[4, :] = [rForest_120.oob_score_, r2V_rForest_120]
rf_table[5, :] = [rForest_240.oob_score_, r2V_rForest_240]


rf_df = pd.DataFrame(rf_table,
                     index=["Default", "No X Randomization", "Min Size(30)", 
                            "Min Size(60)", "Min Size(120)", "Min Size(240)"],
                     columns=["OOB R2", "Validation R2"])

print(rf_df.round(3))

###################################################################
# Boosted Trees

###################################################################
# xgboost needs a data matrix and it's own structure
train_xg = train.values  # training sample
test_xg = test.values  # testing sample

xgb_train = xgb.DMatrix(data=train_xg[:, 1:], label=train_xg[:, 0])
xgb_test = xgb.DMatrix(data=test_xg[:, 1:], label=test_xg[:, 0])

# Boosted tree with default xgboost settings and 500 boosting rounds. 
# Max depth = 6, learning rate = .3
params_default = {'max_depth': 6, 'eta': 0.3, 'objective': 'reg:squarederror', 'eval_metric': 'rmse'}
xgboost_default_cv = xgb.cv(params_default, xgb_train, num_boost_round=500, nfold=5, 
                           verbose_eval=25, seed=42, shuffle=False)
# xgboost's default CV training. Printing output as we go along for illustration

best_iter_default = xgboost_default_cv['test-rmse-mean'].idxmin()
# CV minimizing boosting iteration
xgboost_default = xgb.train(params_default, xgb_train, num_boost_round=500, verbose_eval=0)
# Fit model using all training data

# In sample R²
r2_boost_default_last = calculate_r2_manual(y_train, xgboost_default.predict(xgb_train), y_train_mean)
r2_boost_default_best = calculate_r2_manual(y_train, 
                                           xgboost_default.predict(xgb_train, iteration_range=(0, best_iter_default + 1)), 
                                           y_train_mean)

# Validation R²
r2V_boost_default_last = calculate_r2_manual(y_test, xgboost_default.predict(xgb_test), y_train_mean)
r2V_boost_default_best = calculate_r2_manual(y_test, 
                                            xgboost_default.predict(xgb_test, iteration_range=(0, best_iter_default + 1)), 
                                            y_train_mean)

# Plot CV vs. in-sample fit
if dpl:
    filenm = os.path.join(plotdir, 'BoostDefault.png')
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, 501), xgboost_default_cv['test-rmse-mean'], 'red', linestyle='-', label='Validation')
    # Note: xgboost in Python doesn't store training RMSE in cv by default, so we approximate
    plt.ylim(0, 70000)
    plt.xlabel('Boosting Iteration')
    plt.ylabel('RMSE')
    plt.legend()
    plt.savefig(filenm, dpi=100, bbox_inches='tight')
    plt.close()

### Playing with tuning parameters. 500 boosting rounds
# Max depth = 6, learning rate = .1
params_eta1 = {'max_depth': 6, 'eta': 0.1, 'objective': 'reg:squarederror', 'eval_metric': 'rmse'}
xgboost_eta1_cv = xgb.cv(params_eta1, xgb_train, num_boost_round=500, nfold=5, 
                        verbose_eval=25, seed=42, shuffle=False)
# xgboost's default CV training. Printing output as we go along for illustration.
# Using same folds as in default model.

best_iter_eta1 = xgboost_eta1_cv['test-rmse-mean'].idxmin()
# CV minimizing boosting iteration
xgboost_eta1 = xgb.train(params_eta1, xgb_train, num_boost_round=500, verbose_eval=0)
# Fit model using all training data

# In sample R²
r2_boost_eta1_last = calculate_r2_manual(y_train, xgboost_eta1.predict(xgb_train), y_train_mean)
r2_boost_eta1_best = calculate_r2_manual(y_train, 
                                        xgboost_eta1.predict(xgb_train, iteration_range=(0, best_iter_eta1 + 1)), 
                                        y_train_mean)

# Validation R²
r2V_boost_eta1_last = calculate_r2_manual(y_test, xgboost_eta1.predict(xgb_test), y_train_mean)
r2V_boost_eta1_best = calculate_r2_manual(y_test, 
                                         xgboost_eta1.predict(xgb_test, iteration_range=(0, best_iter_eta1 + 1)), 
                                         y_train_mean)

### Playing with tuning parameters. 1000 boosting rounds
# Max depth = 1, learning rate = .1
params_eta1_d1 = {'max_depth': 1, 'eta': 0.1, 'objective': 'reg:squarederror', 'eval_metric': 'rmse'}
xgboost_eta1_d1_cv = xgb.cv(params_eta1_d1, xgb_train, num_boost_round=1000, nfold=5, 
                           verbose_eval=0, seed=42, shuffle=False)
# xgboost's default CV training. Printing output as we go along for illustration.
# Using same folds as in default model.

best_iter_eta1_d1 = xgboost_eta1_d1_cv['test-rmse-mean'].idxmin()
# CV minimizing boosting iteration
xgboost_eta1_d1 = xgb.train(params_eta1_d1, xgb_train, num_boost_round=1000, verbose_eval=0)
# Fit model using all training data

# In sample R²
r2_boost_eta1_d1_best = calculate_r2_manual(y_train, 
                                           xgboost_eta1_d1.predict(xgb_train, iteration_range=(0, best_iter_eta1_d1 + 1)), 
                                           y_train_mean)

# Validation R²
r2V_boost_eta1_d1_best = calculate_r2_manual(y_test, 
                                            xgboost_eta1_d1.predict(xgb_test, iteration_range=(0, best_iter_eta1_d1 + 1)), 
                                            y_train_mean)

### Playing with tuning parameters. 500 boosting rounds
# Max depth = 2, learning rate = .1
params_eta1_d2 = {'max_depth': 2, 'eta': 0.1, 'objective': 'reg:squarederror', 'eval_metric': 'rmse'}
xgboost_eta1_d2_cv = xgb.cv(params_eta1_d2, xgb_train, num_boost_round=500, nfold=5, 
                           verbose_eval=0, seed=42, shuffle=False)
# xgboost's default CV training. Printing output as we go along for illustration.
# Using same folds as in default model.

best_iter_eta1_d2 = xgboost_eta1_d2_cv['test-rmse-mean'].idxmin()
# CV minimizing boosting iteration
xgboost_eta1_d2 = xgb.train(params_eta1_d2, xgb_train, num_boost_round=500, verbose_eval=0)
# Fit model using all training data

# In sample R²
r2_boost_eta1_d2_best = calculate_r2_manual(y_train, 
                                           xgboost_eta1_d2.predict(xgb_train, iteration_range=(0, best_iter_eta1_d2 + 1)), 
                                           y_train_mean)

# Validation R²
r2V_boost_eta1_d2_best = calculate_r2_manual(y_test, 
                                            xgboost_eta1_d2.predict(xgb_test, iteration_range=(0, best_iter_eta1_d2 + 1)), 
                                            y_train_mean)

### Playing with tuning parameters. 500 boosting rounds
# Max depth = 3, learning rate = .1
params_eta1_d3 = {'max_depth': 3, 'eta': 0.1, 'objective': 'reg:squarederror', 'eval_metric': 'rmse'}
xgboost_eta1_d3_cv = xgb.cv(params_eta1_d3, xgb_train, num_boost_round=500, nfold=5, 
                           verbose_eval=0, seed=42, shuffle=False)
# xgboost's default CV training. Printing output as we go along for illustration.
# Using same folds as in default model.

best_iter_eta1_d3 = xgboost_eta1_d3_cv['test-rmse-mean'].idxmin()
# CV minimizing boosting iteration
xgboost_eta1_d3 = xgb.train(params_eta1_d3, xgb_train, num_boost_round=500, verbose_eval=0)
# Fit model using all training data

# In sample R²
r2_boost_eta1_d3_best = calculate_r2_manual(y_train, 
                                           xgboost_eta1_d3.predict(xgb_train, iteration_range=(0, best_iter_eta1_d3 + 1)), 
                                           y_train_mean)

# Validation R²
r2V_boost_eta1_d3_best = calculate_r2_manual(y_test, 
                                            xgboost_eta1_d3.predict(xgb_test, iteration_range=(0, best_iter_eta1_d3 + 1)), 
                                            y_train_mean)

## Tabulate R² results
boost_table = np.zeros((7, 6))
boost_table[0, :] = [6, 0.3, 500, xgboost_default_cv['test-rmse-mean'].iloc[499], 
                     r2_boost_default_last, r2V_boost_default_last]
boost_table[1, :] = [6, 0.3, best_iter_default, xgboost_default_cv['test-rmse-mean'].min(), 
                     r2_boost_default_best, r2V_boost_default_best]
boost_table[2, :] = [6, 0.1, 500, xgboost_eta1_cv['test-rmse-mean'].iloc[499], 
                     r2_boost_eta1_last, r2V_boost_eta1_last]
boost_table[3, :] = [6, 0.1, best_iter_eta1, xgboost_eta1_cv['test-rmse-mean'].min(), 
                     r2_boost_eta1_best, r2V_boost_eta1_best]
boost_table[4, :] = [1, 0.1, best_iter_eta1_d1, xgboost_eta1_d1_cv['test-rmse-mean'].min(), 
                     r2_boost_eta1_d1_best, r2V_boost_eta1_d1_best]
boost_table[5, :] = [2, 0.1, best_iter_eta1_d2, xgboost_eta1_d2_cv['test-rmse-mean'].min(), 
                     r2_boost_eta1_d2_best, r2V_boost_eta1_d2_best]
boost_table[6, :] = [3, 0.1, best_iter_eta1_d3, xgboost_eta1_d3_cv['test-rmse-mean'].min(), 
                     r2_boost_eta1_d3_best, r2V_boost_eta1_d3_best]

boost_df = pd.DataFrame(boost_table,
                        columns=["Depth", "Rate", "Iter", "CV RMSE", "in-sample R2", "Validation R2"])

# Format the table with appropriate precision
boost_df_formatted = boost_df.copy()
boost_df_formatted["Depth"] = boost_df_formatted["Depth"].astype(int)
boost_df_formatted["Rate"] = boost_df_formatted["Rate"].round(1)
boost_df_formatted["Iter"] = boost_df_formatted["Iter"].astype(int)
boost_df_formatted["CV RMSE"] = boost_df_formatted["CV RMSE"].round(3)
boost_df_formatted["in-sample R2"] = boost_df_formatted["in-sample R2"].round(3)
boost_df_formatted["Validation R2"] = boost_df_formatted["Validation R2"].round(3)

print(boost_df_formatted)

   net_tfa  e401  age    inc  educ  fsize  marr  twoearn  db  pira  hown
0    -3300     0   31  28146    12      5     1        0   0     0     1
1    61010     0   52  32634    16      5     0        0   0     0     1
2     8849     0   50  52206    11      3     1        1   0     1     1
3    -6013     0   28  45252    15      4     1        1   0     0     0
4    -2375     0   42  33126    12      3     0        0   1     0     1
         in-sample  Validation
Depth 1      0.126       0.089
Depth 2      0.236       0.079
Depth 3      0.327       0.082
Big          0.968      -0.644
Validation R² for pruned tree: -0.6442085947665206
Validation R² for random forest with default settings: 0.05179586668648728
Validation R² for random forest with no x randomization: 0.05179586668648728
Validation R² for random forest with min node 30: 0.19608515968791818
Validation R² for random forest with min node 60: 0.18084994613007477
Validation R² for random forest with min node 120: 0.16706694193